# 03 - Train Gravitational Potential (Phi)

Train the potential network $\Phi(x)$ with a frozen distribution function, minimizing the CBE residual.

**Prerequisite**: A completed DF training run (see `02_train_df.ipynb`).

In [ ]:
import jax
print(f"JAX backend: {jax.default_backend()}")
print(f"JAX devices: {jax.devices()}")

In [ ]:
from dpjax.config import load_config, merge_config
from dpjax.paths import PROJECT_ROOT, DATA_DIR, RUNS_DIR

print(f"Project root: {PROJECT_ROOT}")

## 1. Load and Customize Configuration

In [ ]:
cfg = load_config("configs/phi_plummer.yaml")

# Override for a quick test
cfg = merge_config(cfg, {
    "train": {
        "epochs": 4,          # quick test; use 1024 for full training
        "batch_size": 2048,
        "log_every": 20,
        "ckpt_every": 100,
    }
})

import yaml
print(yaml.safe_dump(cfg, sort_keys=False))

## 2. Train Phi

In [ ]:
from experiments.train_phi import run_phi_training

DATA_PATH  = DATA_DIR / "plummer_n131072.h5"
DF_RUN_DIR = RUNS_DIR / "plummer" / "df"
RUN_DIR    = RUNS_DIR / "plummer" / "phi"

result = run_phi_training(
    config=cfg,
    data_path=DATA_PATH,
    df_run_dir=DF_RUN_DIR,
    run_dir=RUN_DIR,
)

print(f"\nTraining complete. Final step: {result['final_step']}")

## 3. Inspect Training Metrics

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import csv

metrics_path = RUN_DIR / "metrics.csv"

with metrics_path.open() as f:
    reader = csv.DictReader(f)
    rows = [r for r in reader]

steps = np.array([float(r["step"]) for r in rows])
loss  = np.array([float(r["loss"]) for r in rows])
r_std = np.array([float(r["residual_std"]) for r in rows])
r_p99 = np.array([float(r["residual_p99_abs"]) for r in rows])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(steps, loss, lw=1.2)
axes[0].set_xlabel("step"); axes[0].set_ylabel("CBE loss")
axes[0].set_title("Loss"); axes[0].grid(True, alpha=0.2)

axes[1].plot(steps, r_std, lw=1.2, color="tab:orange")
axes[1].set_xlabel("step"); axes[1].set_ylabel("residual std")
axes[1].set_title("Residual Std"); axes[1].grid(True, alpha=0.2)

axes[2].plot(steps, r_p99, lw=1.2, color="tab:red")
axes[2].set_xlabel("step"); axes[2].set_ylabel("|residual| p99")
axes[2].set_title("Residual p99"); axes[2].grid(True, alpha=0.2)

fig.tight_layout()
plt.show()

## 4. Quick Evaluation: Radial Phi & Acceleration

In [ ]:
import jax.numpy as jnp
from dpjax.models.potential import grad_phi_apply, phi_apply

phi_model = result["phi_model"]
phi_params = result["phi_params"]
normalizer = result["normalizer"]

std_x = np.asarray(normalizer.std[:3], dtype=np.float32)
mean_x = np.asarray(normalizer.mean[:3], dtype=np.float32)

# Evaluate along x-axis
r = np.geomspace(0.01, 10.0, 256).astype(np.float32)
x_phys = np.stack([r, np.zeros_like(r), np.zeros_like(r)], axis=-1)
x_std = (x_phys - mean_x[None, :]) / std_x[None, :]

phi_learned = np.asarray(phi_apply(phi_model, phi_params, jnp.asarray(x_std)))
grad_phi = np.asarray(grad_phi_apply(phi_model, phi_params, jnp.asarray(x_std)))
ar_learned = -(grad_phi[:, 0] / std_x[0])

# Plummer analytic
phi_true = -(1.0 + r**2)**(-0.5)
ar_true = -r * (1.0 + r**2)**(-1.5)

# Align offset
i_ref = np.argmin(np.abs(r - 1.0))
phi_shift = phi_learned - phi_learned[i_ref] + phi_true[i_ref]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

ax1.plot(r, phi_true, "k-", lw=2, label="Plummer analytic")
ax1.plot(r, phi_shift, "--", lw=1.5, label="Learned (shifted)")
ax1.set_xscale("log"); ax1.set_xlabel("r"); ax1.set_ylabel(r"$\Phi(r)$")
ax1.legend(); ax1.set_title("Potential"); ax1.grid(True, alpha=0.2)

ax2.plot(r, ar_true, "k-", lw=2, label="Plummer analytic")
ax2.plot(r, ar_learned, "--", lw=1.5, label="Learned")
ax2.set_xscale("log"); ax2.set_xlabel("r"); ax2.set_ylabel(r"$a_r(r)$")
ax2.legend(); ax2.set_title("Radial Acceleration"); ax2.grid(True, alpha=0.2)

fig.tight_layout()
plt.show()